# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nileshkushwaha1410/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Rule in plain words:** _(e.g. "Refresh pages that are both stale and high-volume.")_

**Reason code it outputs:** _(ONE code, e.g. STALE_HIGH_VOL, and what it means)_

**Action label:** _(e.g. REFRESH)_

**Signal checks (below): what each flag assumes and what the tables showed.** _(fill in after running)_

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# >>> PASTE the data-load code from your skeleton here (whatever builds `df`), then delete this comment.
assert "df" in globals(), "Paste your skeleton's data-load code so `df` exists"

import json, re, os
import numpy as np, pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

# >>> Change these to YOUR column names (run df.columns to see them).
CFG = dict(
    id_col    = "page_id",             # used in the CSV only; never printed in this notebook
    label_col = "is_declining_label",  # judging only, never a rule input
    stale_col = "days_since_update",   # Signal A (past window only)
    vol_col   = "impressions",         # Signal B (past window only)
)
missing = [v for v in CFG.values() if v not in df.columns]
assert not missing, f"Not in df.columns: {missing}. Fix CFG."
print(f"rows={len(df):,}  base rate of label={df[CFG['label_col']].mean():.3f}")

# Signal checks: one bucket table each (n printed), then a one-word verdict YOU choose.
def bucket_table(d, col, label_col, q=5):
    s = d[[col, label_col]].dropna()
    try:    b = pd.qcut(s[col], q=q, duplicates="drop")
    except ValueError: b = pd.cut(s[col], bins=q)
    t = s.groupby(b, observed=True)[label_col].agg(n="size", rate="mean")
    t["lift_vs_base"] = t["rate"] / s[label_col].mean()
    print(f"{col}: n={len(s):,} (dropped {len(d)-len(s):,} NaN) | base rate={s[label_col].mean():.3f}")
    return t

sigA_col, sigB_col = CFG["stale_col"], CFG["vol_col"]     # <- edit if you use other signals
tA = bucket_table(df, sigA_col, CFG["label_col"]); display(tA)
tB = bucket_table(df, sigB_col, CFG["label_col"]); display(tB)

VERDICT_A = "TODO"   # CONFIRMED | OPPOSITE | MIXED | FALSE
VERDICT_B = "TODO"


AssertionError: Paste your skeleton's data-load code so `df` exists

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

SIGN_A, SIGN_B = +1, +1           # -1 if that signal's verdict was OPPOSITE
REASON_CODE  = "STALE_HIGH_VOL"   # exactly ONE reason code
ACTION_LABEL = "REFRESH"

def pct(s, sign):
    r = s.rank(pct=True)
    return r if sign > 0 else 1 - r

d = df.copy()
d["pctA"] = pct(d[sigA_col], SIGN_A)
d["pctB"] = pct(d[sigB_col], SIGN_B)
d["score"] = d["pctA"] * d["pctB"]                 # 0..1, higher = act first
d = d.dropna(subset=["score"])
d["reason_code"], d["action"] = REASON_CODE, ACTION_LABEL

queue = d.sort_values("score", ascending=False).assign(rank=lambda x: np.arange(1, len(x) + 1))
out_cols = ["rank", CFG["id_col"], "score", "reason_code", "action", sigA_col, sigB_col]
os.makedirs("work/outputs", exist_ok=True)
queue[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)   # label NOT written

K = 50
p_at_k, base = queue.head(K)[CFG["label_col"]].mean(), df[CFG["label_col"]].mean()
metrics = dict(n_rows=int(len(queue)), k=K, precision_at_k=float(p_at_k), base_rate=float(base),
               lift_at_k=float(p_at_k / base), reason_code=REASON_CODE, action=ACTION_LABEL,
               verdict_a=VERDICT_A, verdict_b=VERDICT_B)
json.dump(metrics, open("work/outputs/w04_baseline_metrics.json", "w"), indent=2)
print(metrics)


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top = queue.head(20).copy()
def confidence(r):
    lo = min(r["pctA"], r["pctB"])
    return "higher (both signals in top decile)" if lo >= 0.9 else \
           "medium (both above 0.75)" if lo >= 0.75 else "lower (one signal is only moderate)"
view = pd.DataFrame({
    "rank": top["rank"],
    "row_ref": top.index,                           # row number in df, not the page ID/URL
    "action": top["action"], "reason_code": top["reason_code"],
    "confidence_note": top.apply(confidence, axis=1),
    sigA_col: top[sigA_col], sigB_col: top[sigB_col]})
view

# One line per row: what would make THIS pick wrong? Be specific to that row.
WRONG_IF = {i: "TODO" for i in range(1, 21)}
# Example of the style: WRONG_IF[1] = "Wrong if the volume is a one-week spike, not steady demand."
view["what_would_make_it_wrong"] = view["rank"].map(WRONG_IF)
view


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

# Weak picks: top-50 rows where the label says NOT declining (judging only)
weak = queue.head(50)
weak = weak[weak[CFG["label_col"]] == 0]
print(f"{len(weak)} of the top 50 are label=0")
weak[["rank", sigA_col, sigB_col, "score"]].head(10)

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Leakage check: the rule may only use past-window columns. No label, no future window, no product flags.
rule_inputs = [sigA_col, sigB_col]
print("Rule inputs:", rule_inputs)
bad = [c for c in rule_inputs if re.search(r"label|future|next|target|forward|flag|_y$", c, re.I)]
assert not bad, f"Rule input looks label / future / flag-derived: {bad}"
assert CFG["label_col"] not in rule_inputs
print("Leakage check passed by name. Also confirm by meaning that each input is measured in the past window.")

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.